# 🎮 League of Legends – Sběr dat pro ML (Rank Tier Prediction)

Tento notebook stahuje ranked match data z Riot Games API pro 7 tierů (Iron–Diamond),
ukládá je do SQLite databáze a exportuje finální dataset připravený pro ML klasifikaci.

**Architektura:**
- Rate limiter (100 req / 2 min) + automatický backoff při HTTP 429
- Checkpoint systém – notebook lze bezpečně zastavit a znovu spustit
- Deduplikace matchů přes tabulku `processed_matches`
- Feature engineering přímo při ukládání
- Finální export do CSV

> ⚠️ **BEZPEČNOST:** Nikdy nesdílej svůj Riot API klíč veřejně. Použij Colab Secrets (`userdata.get`).

## 1. Instalace závislostí a konfigurace prostředí

In [1]:
# ── Instalace knihoven (pouze Google Colab prostředí) ──────────────────────────
# sqlite3 a json jsou součástí standardní knihovny Pythonu.
# requests a pandas jsou v Colabu předinstalovány; pro jistotu:
# !pip install requests pandas --quiet

import sqlite3
import json
import time
import logging
import os
import threading
from datetime import datetime, timedelta
from collections import deque
from pathlib import Path

import requests
import pandas as pd

# ── Konfigurace – UPRAV TOTO ───────────────────────────────────────────────────
# Získáš na: https://developer.riotgames.com/  (platnost dev klíče: 24 h)
# V Google Colab použij: from google.colab import userdata; API_KEY = userdata.get('RIOT_API_KEY')

API_KEY: str = "RGAPI-20052a77-2244-4431-a270-1a3c1b83c43a"

REGION_PLATFORM = "euw1"          # platform (summoner/league endpointy)
REGION_CLUSTER  = "europe"        # cluster  (match-v5 endpointy)
QUEUE_ID        = 420             # Ranked Solo/Duo

# Cílové tiery a počet hráčů na tier
TIERS: list[str] = ["IRON", "BRONZE", "SILVER", "GOLD", "PLATINUM", "EMERALD", "DIAMOND"]
DIVISIONS: list[str] = ["I", "II", "III", "IV"]
PLAYERS_PER_TIER: int = 300
MATCHES_PER_PLAYER: int = 20

# Rate limit – Riot dev key: 100 req / 2 min (konzervativně 95 req / 2 min)
RATE_LIMIT_REQUESTS: int = 95
RATE_LIMIT_WINDOW:   int = 120   # sekund

# Cesty k souborům
DB_PATH          = "lol_dataset.db"
CHECKPOINT_PATH  = "checkpoint.json"
LOG_PATH         = "collection.log"
EXPORT_CSV_PATH  = "lol_rank_dataset.csv"

# ── Upozornění na bezpečnost ───────────────────────────────────────────────────
if "xxxxxxxx" in API_KEY:
    print("⚠️  VAROVÁNÍ: Vlož platný Riot API klíč do proměnné API_KEY!")
    print("   Získáš ho na https://developer.riotgames.com/")
    print("   V Colab použij Secrets místo hardcoded klíče.")
else:
    print("✅ Konfigurace načtena.")
    print(f"   Region: {REGION_PLATFORM.upper()} | Tiery: {len(TIERS)} | Hráčů/tier: {PLAYERS_PER_TIER}")
    print(f"   Zápasů/hráč: {MATCHES_PER_PLAYER} | Rate limit: {RATE_LIMIT_REQUESTS} req/{RATE_LIMIT_WINDOW}s")

✅ Konfigurace načtena.
   Region: EUW1 | Tiery: 7 | Hráčů/tier: 300
   Zápasů/hráč: 20 | Rate limit: 95 req/120s


## 2. Inicializace databáze a schéma tabulek

In [2]:
def init_database(db_path: str = DB_PATH) -> sqlite3.Connection:
    """
    Inicializuje SQLite databázi a vytvoří tabulky (IF NOT EXISTS).
    Bezpečné pro opakované spuštění – existující data nebudou smazána.
    """
    conn = sqlite3.connect(db_path, check_same_thread=False)
    conn.execute("PRAGMA journal_mode=WAL")   # lepší souběžný přístup
    conn.execute("PRAGMA synchronous=NORMAL") # kompromis rychlост / bezpečnost

    cursor = conn.cursor()

    # Tabulka hráčů (jeden záznam = jeden summonerName v daném tieru)
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS players (
            id            INTEGER PRIMARY KEY AUTOINCREMENT,
            puuid         TEXT    NOT NULL UNIQUE,
            summonerName  TEXT,
            summonerId    TEXT    UNIQUE,
            tier          TEXT    NOT NULL,
            division      TEXT    NOT NULL,
            leaguePoints  INTEGER DEFAULT 0,
            created_at    TEXT    DEFAULT (datetime('now'))
        )
    """)

    # Tabulka zápasů – základní metadata
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS matches (
            matchId      TEXT PRIMARY KEY,
            queueId      INTEGER,
            gameDuration INTEGER,
            gameVersion  TEXT,
            created_at   TEXT DEFAULT (datetime('now'))
        )
    """)

    # Tabulka participantů – výkon hráče v daném zápase
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS participants (
            id                          INTEGER PRIMARY KEY AUTOINCREMENT,
            matchId                     TEXT    NOT NULL,
            puuid                       TEXT    NOT NULL,
            summonerName                TEXT,
            tier                        TEXT    NOT NULL,
            championName                TEXT,
            role                        TEXT,
            kills                       INTEGER DEFAULT 0,
            deaths                      INTEGER DEFAULT 0,
            assists                     INTEGER DEFAULT 0,
            totalMinionsKilled          INTEGER DEFAULT 0,
            neutralMinionsKilled        INTEGER DEFAULT 0,
            visionScore                 INTEGER DEFAULT 0,
            totalDamageDealtToChampions INTEGER DEFAULT 0,
            totalDamageTaken            INTEGER DEFAULT 0,
            goldEarned                  INTEGER DEFAULT 0,
            timePlayed                  INTEGER DEFAULT 0,
            win                         INTEGER DEFAULT 0,
            -- Odvozené příznaky (feature engineering)
            kda                         REAL    DEFAULT 0.0,
            cs_per_min                  REAL    DEFAULT 0.0,
            damage_per_min              REAL    DEFAULT 0.0,
            gold_per_min                REAL    DEFAULT 0.0,
            deaths_per_min              REAL    DEFAULT 0.0,
            UNIQUE(matchId, puuid)
        )
    """)

    # Tabulka deduplikace – záznamy zpracovaných matchId
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS processed_matches (
            matchId    TEXT PRIMARY KEY,
            processed_at TEXT DEFAULT (datetime('now'))
        )
    """)

    conn.commit()
    print(f"✅ Databáze inicializována: {db_path}")

    # Zobraz aktuální statistiky (pro re-run)
    for table in ["players", "matches", "participants", "processed_matches"]:
        count = cursor.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
        print(f"   {table:<20} → {count:>6} záznamů")

    return conn


# Globální spojení s DB (bude inicializováno při spuštění)
conn: sqlite3.Connection = init_database()

✅ Databáze inicializována: lol_dataset.db
   players              →      0 záznamů
   matches              →      0 záznamů
   participants         →      0 záznamů
   processed_matches    →      0 záznamů


## 3. Rate Limiter & HTTP klient s automatickým backoff

In [3]:
class RateLimiter:
    """Thread-safe sliding-window rate limiter."""

    def __init__(self, max_requests: int = RATE_LIMIT_REQUESTS,
                 window_seconds: int = RATE_LIMIT_WINDOW):
        self.max_requests   = max_requests
        self.window_seconds = window_seconds
        self._timestamps: deque = deque()
        self._lock = threading.Lock()
        self.total_requests: int = 0

    def acquire(self) -> None:
        with self._lock:
            now    = time.monotonic()
            cutoff = now - self.window_seconds
            while self._timestamps and self._timestamps[0] < cutoff:
                self._timestamps.popleft()
            if len(self._timestamps) >= self.max_requests:
                sleep_for = self._timestamps[0] - cutoff + 0.1
                time.sleep(sleep_for)
                now    = time.monotonic()
                cutoff = now - self.window_seconds
                while self._timestamps and self._timestamps[0] < cutoff:
                    self._timestamps.popleft()
            self._timestamps.append(time.monotonic())
            self.total_requests += 1

    @property
    def current_load(self) -> int:
        now = time.monotonic()
        return sum(1 for t in self._timestamps if t >= now - self.window_seconds)


rate_limiter = RateLimiter()

_consecutive_403 = 0
_MAX_CONSECUTIVE_403 = 10


def api_request(url: str, params: dict | None = None,
                max_retries: int = 5) -> dict | list | None:
    global _consecutive_403
    headers = {"X-Riot-Token": API_KEY}
    params  = params or {}

    for attempt in range(1, max_retries + 1):
        try:
            rate_limiter.acquire()
            response = requests.get(url, headers=headers, params=params, timeout=15)

            if response.status_code == 200:
                _consecutive_403 = 0
                return response.json()
            if response.status_code == 429:
                retry_after = int(response.headers.get("Retry-After", 10))
                logging.warning(f"HTTP 429 – čekám {retry_after}s")
                time.sleep(retry_after + 1)
                continue
            if response.status_code in (500, 502, 503, 504):
                time.sleep(2 ** attempt)
                continue
            if response.status_code == 401:
                raise PermissionError("HTTP 401: Neplatný API klíč.")
            if response.status_code == 403:
                _consecutive_403 += 1
                logging.warning(f"HTTP 403 ({_consecutive_403}x) | {url}")
                if _consecutive_403 >= _MAX_CONSECUTIVE_403:
                    raise PermissionError(f"HTTP 403 nastal {_consecutive_403}× za sebou.")
                return None
            if response.status_code == 404:
                return None
            time.sleep(2)

        except PermissionError:
            raise
        except requests.exceptions.Timeout:
            time.sleep(2 ** attempt)
        except requests.exceptions.ConnectionError as e:
            logging.warning(f"Connection error (pokus {attempt}): {e}")
            time.sleep(5)

    return None


def validate_api_key() -> bool:
    """
    Otestuje připojení a platnost API klíče.
    Spouští sérii diagnostických kroků a poradí co opravit.
    """
    print("─" * 55)
    print("DIAGNOSTIKA PŘIPOJENÍ A API KLÍČE")
    print("─" * 55)

    # ── Krok 1: Základní internet ─────────────────────────────────────────────
    print("\n[1/3] Testuji základní internetové připojení …")
    try:
        r = requests.get("https://httpbin.org/get", timeout=8)
        print(f"      ✅ Internet funguje (HTTP {r.status_code})")
        internet_ok = True
    except requests.exceptions.SSLError as e:
        print(f"      ⚠️  SSL chyba – zkouším bez verifikace certifikátu …")
        try:
            r = requests.get("https://httpbin.org/get", timeout=8, verify=False)
            print(f"      ⚠️  Funguje BEZ SSL verifikace – pravděpodobně firemní proxy/VPN.")
            print(f"         Řešení: přidej  verify=False  do volání requests.get().")
            internet_ok = True
        except Exception as e2:
            print(f"      ❌ Nelze se připojit ani bez SSL: {e2}")
            internet_ok = False
    except requests.exceptions.ConnectionError as e:
        print(f"      ❌ Žádné internetové připojení: {e}")
        print()
        print("      Možné příčiny a řešení:")
        print("      • Jsi offline → připoj se k internetu")
        print("      • Firewall blokuje Python → povol python.exe v nastavení firewallu")
        print("        (Windows Defender → Firewall → Povolit aplikaci)")
        print("      • Firemní proxy → nastav v prostředí:")
        print("        import os")
        print("        os.environ['HTTPS_PROXY'] = 'http://proxy:port'")
        print("      • VPN blokuje přímé spojení → zkus vypnout VPN")
        internet_ok = False
    except Exception as e:
        print(f"      ❌ Chyba: {type(e).__name__}: {e}")
        internet_ok = False

    if not internet_ok:
        print("\n🛑 Bez internetu nelze pokračovat.")
        return False

    # ── Krok 2: Riot API dostupnost ───────────────────────────────────────────
    print("\n[2/3] Testuji dosažitelnost Riot API …")
    status_url = f"https://{REGION_PLATFORM}.api.riotgames.com/lol/status/v4/platform-data"
    try:
        r = requests.get(status_url, headers={"X-Riot-Token": API_KEY}, timeout=10)
        if r.status_code == 200:
            print(f"      ✅ Riot API dosažitelné (HTTP 200)")
            riot_ok = True
        elif r.status_code == 401:
            print(f"      ❌ HTTP 401 – API klíč je NEPLATNÝ nebo EXPIROVAL")
            print(f"         Tvůj klíč: {API_KEY[:20]}…")
            print(f"         → Jdi na https://developer.riotgames.com/ → Regenerate API Key")
            return False
        elif r.status_code == 403:
            print(f"      ❌ HTTP 403 – klíč zamítnut i na základní endpoint")
            print(f"         → Vygeneruj nový klíč na https://developer.riotgames.com/")
            return False
        else:
            print(f"      ⚠️  HTTP {r.status_code} – neočekávaná odpověď")
            riot_ok = False
    except requests.exceptions.SSLError:
        print(f"      ❌ SSL chyba při komunikaci s Riot API")
        print(f"         Zkus přidat na začátek buňky 1:")
        print(f"         import urllib3; urllib3.disable_warnings()")
        return False
    except requests.exceptions.ConnectionError as e:
        print(f"      ❌ Nelze se připojit k Riot API: {e}")
        print(f"         IP Riotu mohla být zablokována firewallem/proxí.")
        return False

    # ── Krok 3: League endpoint (odhalí IP blokaci) ───────────────────────────
    print("\n[3/3] Testuji league-v4 endpoint (odhalí Colab/IP blokaci) …")
    league_url = f"https://{REGION_PLATFORM}.api.riotgames.com/lol/league/v4/entries/RANKED_SOLO_5x5/GOLD/I"
    try:
        r = requests.get(league_url, headers={"X-Riot-Token": API_KEY},
                         params={"page": 1}, timeout=10)
        if r.status_code == 200:
            print(f"      ✅ league-v4/entries funguje ({len(r.json())} hráčů vráceno)")
            print()
            print("✅ Vše v pořádku – můžeš spustit sběr dat (buňka 10)!")
            return True
        elif r.status_code == 403:
            print(f"      ❌ HTTP 403 – IP blokace na league endpoint")
            print()
            print("      Status endpoint funguje, ale league endpoint je blokován.")
            print("      Toto nastává z Google Colab (sdílené IP Riotem zakázány).")
            print()
            print("      ╔══════════════════════════════════════════════════════╗")
            print("      ║  Spouštíš VS Code LOKÁLNĚ – domácí IP by měla       ║")
            print("      ║  fungovat. Zkontroluj:                                ║")
            print("      ║                                                        ║")
            print("      ║  1. Vypni VPN (pokud máš zapnutou)                    ║")
            print("      ║  2. Zkus jiné síťové připojení (hotspot místo Wi-Fi)  ║")
            print("      ║  3. Ověř, že python.exe má povolení ve firewallu      ║")
            print("      ╚══════════════════════════════════════════════════════╝")
            return False
        else:
            print(f"      ⚠️  HTTP {r.status_code}")
            return False
    except Exception as e:
        print(f"      ❌ {type(e).__name__}: {e}")
        return False


# ── Spusť diagnostiku ─────────────────────────────────────────────────────────
_key_ok = validate_api_key()
print()
print("✅ RateLimiter a api_request() připraveny." if _key_ok else
      "⚠️  Oprav problém výše a znovu spusť tuto buňku.")

───────────────────────────────────────────────────────
DIAGNOSTIKA PŘIPOJENÍ A API KLÍČE
───────────────────────────────────────────────────────

[1/3] Testuji základní internetové připojení …
      ✅ Internet funguje (HTTP 200)

[2/3] Testuji dosažitelnost Riot API …
      ✅ Riot API dosažitelné (HTTP 200)

[3/3] Testuji league-v4 endpoint (odhalí Colab/IP blokaci) …
      ✅ league-v4/entries funguje (205 hráčů vráceno)

✅ Vše v pořádku – můžeš spustit sběr dat (buňka 10)!

✅ RateLimiter a api_request() připraveny.


## 4. Checkpoint systém – bezpečné přerušení a obnovení

In [4]:
class CheckpointManager:
    """
    Správce checkpointů – ukládá stav průběhu do JSON souboru.

    Struktura checkpoint.json:
    {
        "completed_tiers": ["IRON", "BRONZE", ...],
        "tier_player_counts": {"GOLD": 150, ...},
        "total_matches_downloaded": 4200,
        "started_at": "2025-01-01T12:00:00",
        "last_saved": "2025-01-01T14:30:00"
    }
    """

    DEFAULT_STATE = {
        "completed_tiers":         [],
        "tier_player_counts":      {},
        "total_matches_downloaded": 0,
        "started_at":              None,
        "last_saved":              None,
    }

    def __init__(self, path: str = CHECKPOINT_PATH):
        self.path  = path
        self.state = self._load()

    def _load(self) -> dict:
        """Načte checkpoint ze souboru, případně vrátí prázdný stav."""
        if Path(self.path).exists():
            try:
                with open(self.path, "r", encoding="utf-8") as f:
                    state = json.load(f)
                print(f"✅ Checkpoint načten: {self.path}")
                print(f"   Dokončené tiery: {state.get('completed_tiers', [])}")
                print(f"   Stažené zápasy:  {state.get('total_matches_downloaded', 0)}")
                return state
            except (json.JSONDecodeError, KeyError) as e:
                print(f"⚠️  Checkpoint poškozen ({e}), začínám od nuly.")
        else:
            print("ℹ️  Žádný checkpoint nenalezen – začínám od začátku.")

        state = dict(self.DEFAULT_STATE)
        state["started_at"] = datetime.now().isoformat()
        return state

    def save(self) -> None:
        """Persistentně uloží aktuální stav na disk."""
        self.state["last_saved"] = datetime.now().isoformat()
        with open(self.path, "w", encoding="utf-8") as f:
            json.dump(self.state, f, indent=2, ensure_ascii=False)

    # ── Pohodlné accessory ─────────────────────────────────────────────────────

    def is_tier_completed(self, tier: str) -> bool:
        return tier in self.state["completed_tiers"]

    def mark_tier_completed(self, tier: str) -> None:
        if tier not in self.state["completed_tiers"]:
            self.state["completed_tiers"].append(tier)
        self.save()

    def get_player_count(self, tier: str) -> int:
        return self.state["tier_player_counts"].get(tier, 0)

    def set_player_count(self, tier: str, count: int) -> None:
        self.state["tier_player_counts"][tier] = count
        self.save()

    def add_matches(self, count: int) -> None:
        self.state["total_matches_downloaded"] = \
            self.state.get("total_matches_downloaded", 0) + count
        self.save()

    @property
    def total_matches(self) -> int:
        return self.state.get("total_matches_downloaded", 0)

    def reset(self) -> None:
        """Smaže checkpoint a začne od nuly (použij opatrně!)."""
        self.state = dict(self.DEFAULT_STATE)
        self.state["started_at"] = datetime.now().isoformat()
        self.save()
        print("🔄 Checkpoint resetován.")


# Globální instance
checkpoint = CheckpointManager()

ℹ️  Žádný checkpoint nenalezen – začínám od začátku.


## 5. Riot API helper funkce

### ⚠️ Důležité: HTTP 403 na `league-v4/entries`

Pokud `validate_api_key()` vrátí ✅ ale sběr hráčů vrátí 403, jde o **IP blokaci**.

**Příčina:** Google Colab používá sdílené IP adresy – Riot je má na blocklist.

**3 řešení (od nejjednoduššího):**

| Řešení | Popis |
|--------|-------|
| **1. Colab Pro + nová runtime** | Přiřadí novou IP – zkus `Runtime → Disconnect and delete runtime` a znovu připoj |
| **2. Lokální spuštění** | Spusť notebook lokálně v VS Code (máš ho tam!) – domácí IP není blokována |
| **3. Vlastní server / VPS** | Colab s vlastní runtime nebo Google Cloud VM |

**Rychlý test:** buňka níže obsahuje `test_league_endpoint()` – spusť ji před sběrem.

In [5]:
# ── Základní URL šablony ───────────────────────────────────────────────────────

def _platform_url(path: str) -> str:
    return f"https://{REGION_PLATFORM}.api.riotgames.com{path}"

def _cluster_url(path: str) -> str:
    return f"https://{REGION_CLUSTER}.api.riotgames.com{path}"


# ── Diagnostika: test league endpointu (odhalí IP blokaci) ────────────────────

def test_league_endpoint() -> bool:
    """
    Testuje přístup k league-v4/entries endpointu.
    Pokud vrátí 403 ale status vrátil 200 → IP blokace z Colab.

    Returns:
        True pokud endpoint funguje, False pokud je blokován.
    """
    url = _platform_url("/lol/league/v4/entries/RANKED_SOLO_5x5/GOLD/I")
    headers = {"X-Riot-Token": API_KEY}
    print(f"🔍 Testuji league-v4/entries endpoint …")
    try:
        resp = requests.get(url, headers=headers, params={"page": 1}, timeout=10)
        if resp.status_code == 200:
            data = resp.json()
            print(f"✅ league-v4/entries funguje! Vrátil {len(data)} záznamů.")
            return True
        elif resp.status_code == 403:
            print("❌ HTTP 403 na league-v4/entries – IP blokace!")
            print()
            print("   Toto je ZNÁMÝ problém Google Colab (sdílené IP blokovány Riotem).")
            print()
            print("   ╔══════════════════════════════════════════════════════════╗")
            print("   ║  ŘEŠENÍ:                                                 ║")
            print("   ║  Spusť notebook LOKÁLNĚ ve VS Code místo v Colab!        ║")
            print("   ║                                                           ║")
            print("   ║  1. Otevři tento notebook ve VS Code                     ║")
            print("   ║  2. Vyber Python kernel (Ctrl+Shift+P → Select Kernel)   ║")
            print("   ║  3. Vlož API klíč do buňky 1                             ║")
            print("   ║  4. Run All                                               ║")
            print("   ║                                                           ║")
            print("   ║  Tvá domácí IP není blokována.                            ║")
            print("   ╚══════════════════════════════════════════════════════════╝")
            return False
        elif resp.status_code == 401:
            print("❌ HTTP 401 – API klíč je neplatný.")
            return False
        else:
            print(f"⚠️  HTTP {resp.status_code}")
            return False
    except requests.exceptions.ConnectionError:
        print("❌ Nelze se připojit.")
        return False


# ── 1. Hráči podle tieru a divize (league-v4) ─────────────────────────────────

def get_players_by_tier(tier: str, division: str, page: int = 1) -> list[dict]:
    """
    Endpoint: GET /lol/league/v4/entries/RANKED_SOLO_5x5/{tier}/{division}
    Vrátí max. 205 záznamů na stránku.
    """
    url  = _platform_url(f"/lol/league/v4/entries/RANKED_SOLO_5x5/{tier}/{division}")
    data = api_request(url, params={"page": page})
    return data if isinstance(data, list) else []


# ── 2. Summoner info – PUUID ──────────────────────────────────────────────────

def get_summoner_by_id(summoner_id: str) -> dict | None:
    """
    Endpoint: GET /lol/summoner/v4/summoners/{encryptedSummonerId}
    Vrátí objekt se summonerId, puuid, name, ...
    """
    if not summoner_id or not summoner_id.strip():
        logging.warning("get_summoner_by_id() zavolán s prázdným summonerId – přeskakuji.")
        return None
    url  = _platform_url(f"/lol/summoner/v4/summoners/{summoner_id}")
    data = api_request(url)
    if not data or "puuid" not in data:
        logging.warning(f"Nepodařilo se získat puuid pro summonerId={summoner_id}")
        return None
    return data


# ── 3. Match historie hráče (match-v5) ────────────────────────────────────────

def get_match_history(puuid: str, count: int = MATCHES_PER_PLAYER) -> list[str]:
    """
    Endpoint: GET /lol/match/v5/matches/by-puuid/{puuid}/ids
              ?queue=420&type=ranked&count={count}
    """
    url  = _cluster_url(f"/lol/match/v5/matches/by-puuid/{puuid}/ids")
    data = api_request(url, params={"queue": QUEUE_ID, "type": "ranked", "count": count})
    return data if isinstance(data, list) else []


# ── 4. Detail zápasu ──────────────────────────────────────────────────────────

def get_match_detail(match_id: str) -> dict | None:
    """Endpoint: GET /lol/match/v5/matches/{matchId}"""
    url = _cluster_url(f"/lol/match/v5/matches/{match_id}")
    return api_request(url)


# ── 5. Uložení hráče do DB ────────────────────────────────────────────────────

def upsert_player(puuid: str, summoner_name: str, summoner_id: str,
                  tier: str, division: str, league_points: int) -> None:
    try:
        conn.execute("""
            INSERT INTO players (puuid, summonerName, summonerId, tier, division, leaguePoints)
            VALUES (?, ?, ?, ?, ?, ?)
            ON CONFLICT(puuid) DO UPDATE SET
                summonerName = excluded.summonerName,
                summonerId   = excluded.summonerId,
                tier         = excluded.tier,
                division     = excluded.division,
                leaguePoints = excluded.leaguePoints
        """, (puuid, summoner_name, summoner_id, tier, division, league_points))
    except sqlite3.IntegrityError:
        # Handle summonerId UNIQUE conflict: remove stale record with same summonerId
        conn.execute("DELETE FROM players WHERE summonerId = ? AND puuid != ?",
                     (summoner_id, puuid))
        conn.execute("""
            INSERT INTO players (puuid, summonerName, summonerId, tier, division, leaguePoints)
            VALUES (?, ?, ?, ?, ?, ?)
            ON CONFLICT(puuid) DO UPDATE SET
                summonerName = excluded.summonerName,
                summonerId   = excluded.summonerId,
                tier         = excluded.tier,
                division     = excluded.division,
                leaguePoints = excluded.leaguePoints
        """, (puuid, summoner_name, summoner_id, tier, division, league_points))
    conn.commit()


# ── Spusť test endpointu ───────────────────────────────────────────────────────
print("✅ API helper funkce připraveny.")
print()
league_ok = test_league_endpoint()
if not league_ok:
    print()
    print("💡 Pro lokální spuštění: otevři notebook ve VS Code a spusť buňky zde.")
    print("   Sběr dat LOKÁLNĚ (ne v Colab) bude fungovat bez omezení IP.")

✅ API helper funkce připraveny.

🔍 Testuji league-v4/entries endpoint …
✅ league-v4/entries funguje! Vrátil 205 záznamů.


## 6. Sběr hráčů podle tieru

In [6]:
def collect_players_for_tier(tier: str,
                             target_count: int = PLAYERS_PER_TIER) -> list[str]:
    """
    Sbírá hráče z daného tieru ze všech 4 divizí.

    Algoritmus:
    1. Iteruje přes DIVISIONS (I–IV) a stránky výsledků.
    2. Moderní Riot API vrací puuid přímo v league-v4 záznamu – summoner-v4 se volá
       pouze jako fallback pokud puuid chybí.
    3. Ukládá do tabulky players a loguje průběh.
    4. Respektuje cílový počet hráčů – zastaví se po dosažení target_count.
    5. Přeskočí tier, pokud je označen jako dokončený v checkpointu.

    Args:
        tier:         Název tieru ("IRON", "GOLD", ...)
        target_count: Počet hráčů k sesbírání

    Returns:
        Seznam puuid sebraných hráčů
    """
    log = logging.getLogger(__name__)

    # Kontrola checkpointu – přeskočit hotové tiery
    if checkpoint.is_tier_completed(tier):
        log.info(f"[{tier}] Přeskočen (již dokončen v checkpointu).")
        rows = conn.execute(
            "SELECT puuid FROM players WHERE tier = ?", (tier,)
        ).fetchall()
        return [r[0] for r in rows]

    collected_puuids: list[str] = []
    existing_ids: set[str] = {
        r[0] for r in conn.execute(
            "SELECT summonerId FROM players WHERE tier = ? AND summonerId IS NOT NULL",
            (tier,)
        ).fetchall()
    }

    print(f"\n{'─'*60}")
    print(f"[{tier}] Sbírám hráče | cíl: {target_count} | již v DB: {len(existing_ids)}")

    for division in DIVISIONS:
        if len(collected_puuids) >= target_count:
            break

        page = 1
        while len(collected_puuids) < target_count:
            entries = get_players_by_tier(tier, division, page)
            if not entries:
                log.info(f"[{tier}/{division}] Stránka {page} prázdná – konec stránkování.")
                break

            for entry in entries:
                if len(collected_puuids) >= target_count:
                    break

                summoner_id   = entry.get("summonerId", "") or ""
                summoner_name = entry.get("summonerName", "") or entry.get("riotIdGameName", "")
                lp            = entry.get("leaguePoints", 0)

                # Moderní Riot API vrací puuid přímo v league-v4 záznamu
                puuid = entry.get("puuid", "") or ""

                # Přeskočit duplicity (summonerId již v DB)
                if summoner_id and summoner_id in existing_ids:
                    continue

                # ── Získej puuid ──────────────────────────────────────────────
                if not puuid:
                    # Fallback: zavolej summoner-v4 pouze pokud summonerId není prázdné
                    if not summoner_id:
                        logging.warning("Entry bez puuid i summonerId – přeskakuji.")
                        continue
                    summoner_data = get_summoner_by_id(summoner_id)
                    if summoner_data is None:
                        continue
                    puuid       = summoner_data.get("puuid", "")
                    summoner_id = summoner_data.get("id", summoner_id)
                    if not summoner_name:
                        summoner_name = summoner_data.get("name", "")

                if not puuid:
                    continue

                # Uložit hráče
                upsert_player(puuid, summoner_name, summoner_id,
                              tier, division, lp)
                if summoner_id:
                    existing_ids.add(summoner_id)
                collected_puuids.append(puuid)

                # Logování každých 50 hráčů
                n = len(collected_puuids)
                if n % 50 == 0:
                    print(f"  [{tier}] {n}/{target_count} hráčů sesbíráno")
                    checkpoint.set_player_count(tier, n)

            page += 1

    # Doplnění z DB (pro případ přerušení a re-run)
    if len(collected_puuids) < target_count:
        rows = conn.execute(
            "SELECT puuid FROM players WHERE tier = ?", (tier,)
        ).fetchall()
        collected_puuids = [r[0] for r in rows]

    # Označ tier jako dokončený pokud máme dost hráčů
    if len(collected_puuids) >= target_count:
        checkpoint.mark_tier_completed(tier)

    checkpoint.set_player_count(tier, len(collected_puuids))
    print(f"  [{tier}] ✅ Celkem sesbíráno: {len(collected_puuids)} hráčů")
    return collected_puuids[:target_count]


print("✅ collect_players_for_tier() připravena.")


✅ collect_players_for_tier() připravena.


## 7 & 8. Feature Engineering + Sběr zápasů

In [7]:
def compute_features(p: dict, tier: str, time_played: int) -> dict:
    """
    Vypočítá odvozené příznaky pro jednoho participanta.

    Vzorce:
        kda            = (kills + assists) / max(1, deaths)
        cs_per_min     = (totalMinionsKilled + neutralMinionsKilled) / (timePlayed / 60)
        damage_per_min = totalDamageDealtToChampions / (timePlayed / 60)
        gold_per_min   = goldEarned / (timePlayed / 60)
        deaths_per_min = deaths / (timePlayed / 60)

    Args:
        p:           Participant dict z match-v5 response
        tier:        Tier hráče (přiřazený ze seznamu, ne z API)
        time_played: Délka hry v sekundách

    Returns:
        Dict s raw atributy + odvozenými příznaky
    """
    kills   = int(p.get("kills", 0))
    deaths  = int(p.get("deaths", 0))
    assists = int(p.get("assists", 0))

    total_cs    = int(p.get("totalMinionsKilled", 0)) + int(p.get("neutralMinionsKilled", 0))
    total_dmg   = int(p.get("totalDamageDealtToChampions", 0))
    gold        = int(p.get("goldEarned", 0))

    # Bezpečné dělení minutami (ošetření timePlayed=0)
    minutes = max(time_played / 60, 1e-6)

    return {
        "puuid":                       p.get("puuid", ""),
        "summonerName":                p.get("summonerName", p.get("riotIdGameName", "")),
        "tier":                        tier,
        "championName":                p.get("championName", ""),
        "role":                        p.get("teamPosition", p.get("role", "")),
        "kills":                       kills,
        "deaths":                      deaths,
        "assists":                     assists,
        "totalMinionsKilled":          int(p.get("totalMinionsKilled", 0)),
        "neutralMinionsKilled":        int(p.get("neutralMinionsKilled", 0)),
        "visionScore":                 int(p.get("visionScore", 0)),
        "totalDamageDealtToChampions": total_dmg,
        "totalDamageTaken":            int(p.get("totalDamageTaken", 0)),
        "goldEarned":                  gold,
        "timePlayed":                  time_played,
        "win":                         1 if p.get("win", False) else 0,
        # ── Odvozené příznaky ──────────────────────────────────────────────────
        "kda":            round((kills + assists) / max(1, deaths), 4),
        "cs_per_min":     round(total_cs    / minutes, 4),
        "damage_per_min": round(total_dmg   / minutes, 4),
        "gold_per_min":   round(gold        / minutes, 4),
        "deaths_per_min": round(deaths      / minutes, 4),
    }


def collect_matches_for_player(puuid: str, tier: str) -> int:
    """
    Sbírá match detaily pro jednoho hráče.

    1. Stáhne posledních MATCHES_PER_PLAYER matchId.
    2. Zkontroluje processed_matches pro deduplikaci.
    3. Stáhne detail pouze nových zápasů.
    4. Uloží metadata do matches a výkony do participants.
    5. Zaznamená matchId do processed_matches atomicky.

    Returns:
        Počet nově uložených zápasů.
    """
    match_ids = get_match_history(puuid)
    if not match_ids:
        return 0

    new_matches = 0
    for match_id in match_ids:
        # ── Deduplikace ────────────────────────────────────────────────────────
        already = conn.execute(
            "SELECT 1 FROM processed_matches WHERE matchId = ?", (match_id,)
        ).fetchone()
        if already:
            continue  # Zápas již zpracován – přeskočit

        # ── Stáhni detail zápasu ──────────────────────────────────────────────
        detail = get_match_detail(match_id)
        if not detail:
            continue

        info     = detail.get("info", {})
        metadata = detail.get("metadata", {})

        game_duration = int(info.get("gameDuration", 0))
        game_version  = info.get("gameVersion", "")
        queue_id      = int(info.get("queueId", 0))

        # ── Ulož match metadata ────────────────────────────────────────────────
        try:
            conn.execute("""
                INSERT OR IGNORE INTO matches (matchId, queueId, gameDuration, gameVersion)
                VALUES (?, ?, ?, ?)
            """, (match_id, queue_id, game_duration, game_version))
        except sqlite3.Error as e:
            logging.warning(f"Chyba při ukládání match {match_id}: {e}")
            continue

        # ── Zpracuj participants ───────────────────────────────────────────────
        participants = info.get("participants", [])
        for p in participants:
            # Pouze hráč, jehož puuid odpovídá aktuálnímu hráči, dostane tier.
            # Ostatní participanti v zápase dostanou tier "UNKNOWN" (neznáme jejich rank).
            p_puuid      = p.get("puuid", "")
            p_tier       = tier if p_puuid == puuid else "UNKNOWN"
            p_time_played = int(p.get("timePlayed", game_duration))

            features = compute_features(p, p_tier, p_time_played)

            try:
                conn.execute("""
                    INSERT OR IGNORE INTO participants
                        (matchId, puuid, summonerName, tier, championName, role,
                         kills, deaths, assists, totalMinionsKilled, neutralMinionsKilled,
                         visionScore, totalDamageDealtToChampions, totalDamageTaken,
                         goldEarned, timePlayed, win,
                         kda, cs_per_min, damage_per_min, gold_per_min, deaths_per_min)
                    VALUES
                        (?, ?, ?, ?, ?, ?,
                         ?, ?, ?, ?, ?,
                         ?, ?, ?,
                         ?, ?, ?,
                         ?, ?, ?, ?, ?)
                """, (
                    match_id,
                    features["puuid"], features["summonerName"], features["tier"],
                    features["championName"], features["role"],
                    features["kills"], features["deaths"], features["assists"],
                    features["totalMinionsKilled"], features["neutralMinionsKilled"],
                    features["visionScore"], features["totalDamageDealtToChampions"],
                    features["totalDamageTaken"], features["goldEarned"],
                    features["timePlayed"], features["win"],
                    features["kda"], features["cs_per_min"],
                    features["damage_per_min"], features["gold_per_min"],
                    features["deaths_per_min"],
                ))
            except sqlite3.Error as e:
                logging.warning(f"Chyba participant {p_puuid} v {match_id}: {e}")

        # ── Atomicky označ match jako zpracovaný ──────────────────────────────
        conn.execute(
            "INSERT OR IGNORE INTO processed_matches (matchId) VALUES (?)", (match_id,)
        )
        conn.commit()
        new_matches += 1

    return new_matches


print("✅ compute_features() a collect_matches_for_player() připraveny.")

✅ compute_features() a collect_matches_for_player() připraveny.


## 9. Logování průběhu, odhad requestů a době běhu

In [8]:
class ProgressTracker:
    """
    Loguje zprávy do konzole i do souboru s časovým razítkem.
    Uchovává historii zpráv pro souhrnný výpis na konci.
    """

    def __init__(self, log_path: str = LOG_PATH):
        self.log_path = log_path
        self._messages: list[str] = []

        # Nastavení Python logging (file + console)
        logging.basicConfig(
            level=logging.INFO,
            format="%(asctime)s [%(levelname)s] %(message)s",
            handlers=[
                logging.FileHandler(log_path, encoding="utf-8"),
                logging.StreamHandler(),
            ],
            force=True,
        )
        self.logger = logging.getLogger("LoLCollector")

    def info(self, msg: str) -> None:
        ts = datetime.now().strftime("%H:%M:%S")
        formatted = f"[{ts}] {msg}"
        self._messages.append(formatted)
        self.logger.info(msg)

    def warning(self, msg: str) -> None:
        self.logger.warning(msg)

    def error(self, msg: str) -> None:
        self.logger.error(msg)

    def summary(self) -> None:
        """Vypíše posledních 20 zpráv a celkový stav DB."""
        print("\n" + "═"*60)
        print("SOUHRN PRŮBĚHU (posledních 20 zpráv):")
        for m in self._messages[-20:]:
            print(" ", m)
        print("═"*60)


progress = ProgressTracker()


# ── Odhad requestů a doby běhu ────────────────────────────────────────────────

def estimate_runtime(
    n_tiers:         int = len(TIERS),
    players_per_tier: int = PLAYERS_PER_TIER,
    matches_per_player: int = MATCHES_PER_PLAYER,
    rate_req:        int = RATE_LIMIT_REQUESTS,
    rate_window:     int = RATE_LIMIT_WINDOW,
) -> None:
    """
    Vytiskne tabulku s odhadem počtu requestů a předpokládanou dobou běhu.

    Výpočet na tier:
        pages_per_division ≈ ceil(players_per_tier / (4 * 200))   (200 zázn/stránka)
        req_league    = 4 divisions × pages (stahování žebříčku)
        req_summoner  = players_per_tier  (1 req / hráč pro puuid)
        req_history   = players_per_tier  (1 req / hráč pro matchId)
        req_details   = players_per_tier × matches_per_player (1 req / zápas)
    """
    import math

    pages_per_division = max(1, math.ceil(players_per_tier / (4 * 200)))
    req_league   = 4 * pages_per_division
    req_summoner = players_per_tier
    req_history  = players_per_tier
    req_details  = players_per_tier * matches_per_player

    req_per_tier = req_league + req_summoner + req_history + req_details
    total_req    = n_tiers * req_per_tier

    # Teoretická minimální doba: celkový čas = (total_req / rate_req) * rate_window
    min_seconds  = (total_req / rate_req) * rate_window
    min_minutes  = min_seconds / 60
    min_hours    = min_minutes  / 60

    header = f"{'Tier':<12} {'League':<8} {'Summoner':<10} {'History':<9} {'Details':<10} {'Celkem'}"
    print("\n" + "═"*70)
    print("ODHAD POČTU REQUESTŮ A DOBY BĚHU")
    print("═"*70)
    print(header)
    print("─"*70)

    for tier in TIERS:
        print(f"{tier:<12} {req_league:<8} {req_summoner:<10} {req_history:<9} {req_details:<10} {req_per_tier}")

    print("─"*70)
    print(f"{'CELKEM':<12} {req_league*n_tiers:<8} {req_summoner*n_tiers:<10} "
          f"{req_history*n_tiers:<9} {req_details*n_tiers:<10} {total_req}")
    print(f"\nRate limit: {rate_req} req / {rate_window}s")
    print(f"Odhadovaná minimální doba běhu:")
    print(f"  ≈ {total_req:,} req ÷ {rate_req} req/{rate_window}s")
    print(f"  ≈ {min_minutes:,.0f} minut")
    print(f"  ≈ {min_hours:.1f} hodin")
    print(f"\n⚠️  Reálná doba bude delší kvůli latenci API, retry logice a network overhead.")
    print("═"*70)


# Spusť odhad
estimate_runtime()


══════════════════════════════════════════════════════════════════════
ODHAD POČTU REQUESTŮ A DOBY BĚHU
══════════════════════════════════════════════════════════════════════
Tier         League   Summoner   History   Details    Celkem
──────────────────────────────────────────────────────────────────────
IRON         4        300        300       6000       6604
BRONZE       4        300        300       6000       6604
SILVER       4        300        300       6000       6604
GOLD         4        300        300       6000       6604
PLATINUM     4        300        300       6000       6604
EMERALD      4        300        300       6000       6604
DIAMOND      4        300        300       6000       6604
──────────────────────────────────────────────────────────────────────
CELKEM       28       2100       2100      42000      46228

Rate limit: 95 req / 120s
Odhadovaná minimální doba běhu:
  ≈ 46,228 req ÷ 95 req/120s
  ≈ 973 minut
  ≈ 16.2 hodin

⚠️  Reálná doba bude delší kvů

## 10. Hlavní smyčka – orchestrace sběru dat

> Spusť tuto buňku pro zahájení (nebo pokračování) sběru dat.
> Notebook lze kdykoliv přerušit klávesou **Interrupt** – při dalším spuštění pokračuje od checkpointu.

In [9]:
def collect_all_data(
    tiers:             list[str] = TIERS,
    players_per_tier:  int       = PLAYERS_PER_TIER,
    matches_per_player: int      = MATCHES_PER_PLAYER,
) -> None:
    """
    Hlavní orchestrační funkce.

    Pro každý tier:
      1. Sesbírá players_per_tier hráčů.
      2. Pro každého hráče stáhne posledních matches_per_player zápasů.
      3. Průběžně aktualizuje checkpoint.
      4. Na konci každého tieru vytiskne souhrn.

    Při přerušení (KeyboardInterrupt) bezpečně uloží checkpoint a uzavře DB.
    """
    import sys

    start_time = time.time()
    progress.info("═"*60)
    progress.info("SBĚR DAT ZAHÁJEN")
    progress.info(f"Tiery: {tiers}")
    progress.info(f"Hráčů/tier: {players_per_tier} | Zápasů/hráč: {matches_per_player}")
    progress.info("═"*60)

    try:
        for tier_idx, tier in enumerate(tiers, 1):
            tier_start = time.time()

            # ── 1. Načti hráče ─────────────────────────────────────────────────
            progress.info(f"[{tier_idx}/{len(tiers)}] {tier} – sbírám hráče...")
            puuids = collect_players_for_tier(tier, players_per_tier)

            if not puuids:
                progress.warning(f"[{tier}] Žádní hráči nenalezeni – přeskakuji.")
                continue

            progress.info(f"[{tier}] {len(puuids)} hráčů | sbírám zápasy...")

            # ── 2. Stáhni zápasy ───────────────────────────────────────────────
            tier_new_matches = 0
            for player_idx, puuid in enumerate(puuids, 1):
                player_start = time.time()
                new = collect_matches_for_player(puuid, tier)
                player_elapsed = time.time() - player_start
                tier_new_matches += new
                checkpoint.add_matches(new)

                # Log KAŽDÉHO hráče – aby bylo vidět, že skript běží
                elapsed = time.time() - start_time
                print(
                    f"  [{tier}] hráč {player_idx}/{len(puuids)} | "
                    f"+{new} zápasů ({player_elapsed:.1f}s) | "
                    f"⏱ {elapsed/60:.1f}min | "
                    f"req: {rate_limiter.total_requests}",
                    flush=True
                )

                # Podrobnější souhrn každých 25 hráčů
                if player_idx % 25 == 0:
                    total_in_db = conn.execute(
                        "SELECT COUNT(*) FROM participants WHERE tier = ?", (tier,)
                    ).fetchone()[0]
                    avg_per_player = tier_new_matches / player_idx
                    remaining = len(puuids) - player_idx
                    eta_min = (remaining * (elapsed / player_idx)) / 60
                    progress.info(
                        f"  ── [{tier}] SOUHRN {player_idx}/{len(puuids)} ──  "
                        f"v DB: {total_in_db} | "
                        f"∅ {avg_per_player:.1f} zápasů/hráč | "
                        f"ETA tier: ~{eta_min:.0f}min"
                    )

            # ── 3. Tier dokončen ───────────────────────────────────────────────
            tier_elapsed = time.time() - tier_start
            total_participants = conn.execute(
                "SELECT COUNT(*) FROM participants WHERE tier != 'UNKNOWN'"
            ).fetchone()[0]

            progress.info(
                f"✅ [{tier}] DOKONČEN | "
                f"nové zápasy: {tier_new_matches} | "
                f"čas tieru: {tier_elapsed/60:.1f}min | "
                f"celkem participants (labeled): {total_participants}"
            )
            checkpoint.mark_tier_completed(tier)

    except KeyboardInterrupt:
        progress.warning("⏸ Sběr přerušen uživatelem – checkpoint uložen.")
        checkpoint.save()
    finally:
        conn.commit()

    # ── Závěrečný souhrn ───────────────────────────────────────────────────────
    total_elapsed = time.time() - start_time
    total_participants = conn.execute("SELECT COUNT(*) FROM participants").fetchone()[0]
    total_labeled      = conn.execute(
        "SELECT COUNT(*) FROM participants WHERE tier != 'UNKNOWN'"
    ).fetchone()[0]
    total_matches_db   = conn.execute("SELECT COUNT(*) FROM matches").fetchone()[0]

    progress.info("═"*60)
    progress.info("SBĚR DAT DOKONČEN")
    progress.info(f"  Celkový čas:          {total_elapsed/3600:.2f} hodin")
    progress.info(f"  API requesty celkem:  {rate_limiter.total_requests}")
    progress.info(f"  Zápasy v DB:          {total_matches_db}")
    progress.info(f"  Participants (total): {total_participants}")
    progress.info(f"  Labeled participants: {total_labeled}")
    progress.info("═"*60)

    # Distribuce dat přes tiery
    print("\nDistribuce labeled participants podle tieru:")
    rows = conn.execute("""
        SELECT tier, COUNT(*) as cnt
        FROM participants
        WHERE tier != 'UNKNOWN'
        GROUP BY tier
        ORDER BY CASE tier
            WHEN 'IRON'     THEN 1
            WHEN 'BRONZE'   THEN 2
            WHEN 'SILVER'   THEN 3
            WHEN 'GOLD'     THEN 4
            WHEN 'PLATINUM' THEN 5
            WHEN 'EMERALD'  THEN 6
            WHEN 'DIAMOND'  THEN 7
            ELSE 8 END
    """).fetchall()
    for tier, cnt in rows:
        bar = "█" * min(50, cnt // 100)
        print(f"  {tier:<12} {cnt:>6}  {bar}")


# ─────────────────────────────────────────────────────────────────────────────
# HLAVNÍ VSTUPNÍ BOD – spusť sběr dat
# ─────────────────────────────────────────────────────────────────────────────
if "xxxxxxxx" in API_KEY or not API_KEY:
    print("⚠️  Nelze spustit – vlož platný Riot API klíč do proměnné API_KEY (buňka 1).")
elif not validate_api_key():
    print("\n🛑 Sběr dat ZASTAVEN – oprav API klíč a spusť znovu.")
    print("   1. Jdi na https://developer.riotgames.com/")
    print("   2. Vygeneruj nový dev klíč (platí 24 h)")
    print("   3. Vlož ho do buňky 1 a znovu spusť buňky 1 → 10")
else:
    collect_all_data()

───────────────────────────────────────────────────────
DIAGNOSTIKA PŘIPOJENÍ A API KLÍČE
───────────────────────────────────────────────────────

[1/3] Testuji základní internetové připojení …
      ✅ Internet funguje (HTTP 200)

[2/3] Testuji dosažitelnost Riot API …
      ✅ Riot API dosažitelné (HTTP 200)

[3/3] Testuji league-v4 endpoint (odhalí Colab/IP blokaci) …


2026-03-04 21:15:45,712 [INFO] ════════════════════════════════════════════════════════════
2026-03-04 21:15:45,716 [INFO] SBĚR DAT ZAHÁJEN
2026-03-04 21:15:45,718 [INFO] Tiery: ['IRON', 'BRONZE', 'SILVER', 'GOLD', 'PLATINUM', 'EMERALD', 'DIAMOND']
2026-03-04 21:15:45,719 [INFO] Hráčů/tier: 300 | Zápasů/hráč: 20
2026-03-04 21:15:45,721 [INFO] ════════════════════════════════════════════════════════════
2026-03-04 21:15:45,722 [INFO] [1/7] IRON – sbírám hráče...


      ✅ league-v4/entries funguje (205 hráčů vráceno)

✅ Vše v pořádku – můžeš spustit sběr dat (buňka 10)!

────────────────────────────────────────────────────────────
[IRON] Sbírám hráče | cíl: 300 | již v DB: 0
  [IRON] 50/300 hráčů sesbíráno
  [IRON] 100/300 hráčů sesbíráno
  [IRON] 150/300 hráčů sesbíráno
  [IRON] 200/300 hráčů sesbíráno


2026-03-04 21:15:46,697 [INFO] [IRON] 300 hráčů | sbírám zápasy...


  [IRON] 250/300 hráčů sesbíráno
  [IRON] 300/300 hráčů sesbíráno
  [IRON] ✅ Celkem sesbíráno: 300 hráčů
  [IRON] hráč 1/300 | +20 zápasů (9.3s) | ⏱ 0.2min | req: 23
  [IRON] hráč 2/300 | +20 zápasů (5.2s) | ⏱ 0.3min | req: 44
  [IRON] hráč 3/300 | +20 zápasů (4.9s) | ⏱ 0.3min | req: 65
  [IRON] hráč 4/300 | +20 zápasů (7.0s) | ⏱ 0.5min | req: 86
  [IRON] hráč 5/300 | +20 zápasů (97.8s) | ⏱ 2.1min | req: 107
  [IRON] hráč 6/300 | +20 zápasů (8.0s) | ⏱ 2.2min | req: 128
  [IRON] hráč 7/300 | +20 zápasů (5.2s) | ⏱ 2.3min | req: 149
  [IRON] hráč 8/300 | +20 zápasů (5.9s) | ⏱ 2.4min | req: 170
  [IRON] hráč 9/300 | +20 zápasů (96.3s) | ⏱ 4.0min | req: 191
  [IRON] hráč 10/300 | +20 zápasů (9.8s) | ⏱ 4.2min | req: 212
  [IRON] hráč 11/300 | +20 zápasů (5.4s) | ⏱ 4.3min | req: 233
  [IRON] hráč 12/300 | +20 zápasů (5.5s) | ⏱ 4.4min | req: 254
  [IRON] hráč 13/300 | +20 zápasů (6.5s) | ⏱ 4.5min | req: 275
  [IRON] hráč 14/300 | +20 zápasů (98.0s) | ⏱ 6.1min | req: 296
  [IRON] hráč 15/300 | 

...

  [DIAMOND] hráč 251/300 | +20 zápasů (19.3s) | ⏱ 867.1min | req: 41160
  [DIAMOND] hráč 252/300 | +20 zápasů (16.7s) | ⏱ 867.4min | req: 41181
  [DIAMOND] hráč 253/300 | +20 zápasů (16.7s) | ⏱ 867.7min | req: 41202
  [DIAMOND] hráč 254/300 | +20 zápasů (28.8s) | ⏱ 868.2min | req: 41223
  [DIAMOND] hráč 255/300 | +20 zápasů (49.0s) | ⏱ 869.0min | req: 41244
  [DIAMOND] hráč 256/300 | +18 zápasů (14.8s) | ⏱ 869.2min | req: 41263
  [DIAMOND] hráč 257/300 | +20 zápasů (18.2s) | ⏱ 869.5min | req: 41284
  [DIAMOND] hráč 258/300 | +19 zápasů (14.1s) | ⏱ 869.8min | req: 41304
  [DIAMOND] hráč 259/300 | +20 zápasů (31.4s) | ⏱ 870.3min | req: 41325
  [DIAMOND] hráč 260/300 | +20 zápasů (47.8s) | ⏱ 871.1min | req: 41346
  [DIAMOND] hráč 261/300 | +20 zápasů (16.4s) | ⏱ 871.4min | req: 41367
  [DIAMOND] hráč 262/300 | +20 zápasů (16.5s) | ⏱ 871.6min | req: 41388
  [DIAMOND] hráč 263/300 | +19 zápasů (27.6s) | ⏱ 872.1min | req: 41408
  [DIAMOND] hráč 264/300 | +20 zápasů (47.6s) | ⏱ 872.9min | req

2026-03-05 11:53:12,608 [INFO]   ── [DIAMOND] SOUHRN 275/300 ──  v DB: 5349 | ∅ 19.5 zápasů/hráč | ETA tier: ~80min


  [DIAMOND] hráč 276/300 | +20 zápasů (16.2s) | ⏱ 877.7min | req: 41679
  [DIAMOND] hráč 277/300 | +19 zápasů (28.2s) | ⏱ 878.2min | req: 41699
  [DIAMOND] hráč 278/300 | +20 zápasů (49.1s) | ⏱ 879.0min | req: 41720
  [DIAMOND] hráč 279/300 | +20 zápasů (15.9s) | ⏱ 879.3min | req: 41741
  [DIAMOND] hráč 280/300 | +20 zápasů (18.0s) | ⏱ 879.6min | req: 41762
  [DIAMOND] hráč 281/300 | +20 zápasů (29.4s) | ⏱ 880.1min | req: 41783
  [DIAMOND] hráč 282/300 | +20 zápasů (45.7s) | ⏱ 880.8min | req: 41804
  [DIAMOND] hráč 283/300 | +20 zápasů (19.6s) | ⏱ 881.2min | req: 41825
  [DIAMOND] hráč 284/300 | +20 zápasů (16.6s) | ⏱ 881.4min | req: 41846
  [DIAMOND] hráč 285/300 | +19 zápasů (15.3s) | ⏱ 881.7min | req: 41866
  [DIAMOND] hráč 286/300 | +19 zápasů (28.8s) | ⏱ 882.2min | req: 41886
  [DIAMOND] hráč 287/300 | +19 zápasů (47.6s) | ⏱ 883.0min | req: 41906
  [DIAMOND] hráč 288/300 | +20 zápasů (16.2s) | ⏱ 883.2min | req: 41927
  [DIAMOND] hráč 289/300 | +20 zápasů (19.5s) | ⏱ 883.6min | req

2026-03-05 12:04:00,164 [INFO]   ── [DIAMOND] SOUHRN 300/300 ──  v DB: 5841 | ∅ 19.5 zápasů/hráč | ETA tier: ~0min
2026-03-05 12:04:00,260 [INFO] ✅ [DIAMOND] DOKONČEN | nové zápasy: 5844 | čas tieru: 129.2min | celkem participants (labeled): 40038
2026-03-05 12:04:00,420 [INFO] ════════════════════════════════════════════════════════════
2026-03-05 12:04:00,421 [INFO] SBĚR DAT DOKONČEN
2026-03-05 12:04:00,422 [INFO]   Celkový čas:          14.80 hodin
2026-03-05 12:04:00,422 [INFO]   API requesty celkem:  42176
2026-03-05 12:04:00,422 [INFO]   Zápasy v DB:          40062
2026-03-05 12:04:00,423 [INFO]   Participants (total): 400380
2026-03-05 12:04:00,423 [INFO]   Labeled participants: 40038
2026-03-05 12:04:00,424 [INFO] ════════════════════════════════════════════════════════════



Distribuce labeled participants podle tieru:
  IRON           5231  ██████████████████████████████████████████████████
  BRONZE         5644  ██████████████████████████████████████████████████
  SILVER         5792  ██████████████████████████████████████████████████
  GOLD           5816  ██████████████████████████████████████████████████
  PLATINUM       5858  ██████████████████████████████████████████████████
  EMERALD        5856  ██████████████████████████████████████████████████
  DIAMOND        5841  ██████████████████████████████████████████████████


## 11. Export finálního datasetu do CSV pro ML

In [10]:
# Ordinální kódování tieru pro ML (cílová proměnná)
TIER_ENCODING: dict[str, int] = {
    "IRON":     0,
    "BRONZE":   1,
    "SILVER":   2,
    "GOLD":     3,
    "PLATINUM": 4,
    "EMERALD":  5,
    "DIAMOND":  6,
}

# Sloupce pro ML dataset (feature set + target)
ML_FEATURE_COLUMNS = [
    # Identifikátory
    "matchId", "puuid", "summonerName", "championName", "role",
    # Raw statistiky
    "kills", "deaths", "assists",
    "totalMinionsKilled", "neutralMinionsKilled",
    "visionScore",
    "totalDamageDealtToChampions", "totalDamageTaken",
    "goldEarned", "timePlayed", "win",
    # Odvozené příznaky
    "kda", "cs_per_min", "damage_per_min", "gold_per_min", "deaths_per_min",
    # Cílová proměnná
    "tier", "tier_encoded",
]


def export_ml_dataset(output_path: str = EXPORT_CSV_PATH,
                      db_path:     str = DB_PATH) -> pd.DataFrame:
    """
    Exportuje finální ML dataset z databáze do CSV souboru.

    Pro každý řádek:
      - Jeden hráč v jednom zápase (labeled – tier != UNKNOWN)
      - Všechny raw i odvozené příznaky
      - Cílová proměnná: tier (string) + tier_encoded (int 0–6)

    Po exportu odstraní řádky s NULL hodnotami.
    Vytiskne tvar datasetu a distribuci tříd.

    Returns:
        pandas DataFrame finálního datasetu
    """
    export_conn = sqlite3.connect(db_path)

    print("📦 Načítám data z databáze...")

    query = """
        SELECT
            p.matchId,
            p.puuid,
            p.summonerName,
            p.championName,
            p.role,
            p.kills,
            p.deaths,
            p.assists,
            p.totalMinionsKilled,
            p.neutralMinionsKilled,
            p.visionScore,
            p.totalDamageDealtToChampions,
            p.totalDamageTaken,
            p.goldEarned,
            p.timePlayed,
            p.win,
            p.kda,
            p.cs_per_min,
            p.damage_per_min,
            p.gold_per_min,
            p.deaths_per_min,
            p.tier,
            m.gameDuration,
            m.gameVersion
        FROM participants p
        LEFT JOIN matches m ON p.matchId = m.matchId
        WHERE p.tier != 'UNKNOWN'
          AND p.tier IS NOT NULL
    """

    df = pd.read_sql_query(query, export_conn)
    export_conn.close()

    print(f"  Načteno řádků: {len(df):,}")

    # ── Kódování tieru ─────────────────────────────────────────────────────────
    df["tier_encoded"] = df["tier"].map(TIER_ENCODING)

    # ── Odstranění řádků s NULL hodnotami v feature sloupcích ─────────────────
    feature_cols = [
        "kills", "deaths", "assists", "totalMinionsKilled", "neutralMinionsKilled",
        "visionScore", "totalDamageDealtToChampions", "totalDamageTaken",
        "goldEarned", "timePlayed", "kda", "cs_per_min", "damage_per_min",
        "gold_per_min", "deaths_per_min", "tier_encoded"
    ]
    before = len(df)
    df.dropna(subset=feature_cols, inplace=True)
    after  = len(df)
    if before != after:
        print(f"  ⚠️  Odstraněno {before - after} řádků s NULL hodnotami.")

    # ── Odstranění hráčů s timePlayed = 0 (rematch, error) ────────────────────
    df = df[df["timePlayed"] > 60]  # min. 1 minuta

    # ── Export do CSV ─────────────────────────────────────────────────────────
    df.to_csv(output_path, index=False, encoding="utf-8")

    # ── Souhrn ─────────────────────────────────────────────────────────────────
    print(f"\n✅ Dataset exportován: {output_path}")
    print(f"   Tvar datasetu:   {df.shape[0]:,} řádků × {df.shape[1]} sloupců")
    print(f"   Cílová proměnná: tier_encoded (0=Iron … 6=Diamond)")
    print(f"\nDistribuce tříd (tier):")
    counts = df["tier"].value_counts()
    for tier, cnt in sorted(counts.items(),
                            key=lambda x: TIER_ENCODING.get(x[0], 99)):
        enc = TIER_ENCODING.get(tier, "?")
        bar  = "█" * min(40, cnt // max(1, len(df) // 400))
        pct  = cnt / len(df) * 100
        print(f"  [{enc}] {tier:<12} {cnt:>6}  ({pct:.1f}%)  {bar}")

    return df


# Spusť export
df_ml = export_ml_dataset()

📦 Načítám data z databáze...
  Načteno řádků: 40,038

✅ Dataset exportován: lol_rank_dataset.csv
   Tvar datasetu:   40,036 řádků × 25 sloupců
   Cílová proměnná: tier_encoded (0=Iron … 6=Diamond)

Distribuce tříd (tier):
  [0] IRON           5230  (13.1%)  ████████████████████████████████████████
  [1] BRONZE         5644  (14.1%)  ████████████████████████████████████████
  [2] SILVER         5792  (14.5%)  ████████████████████████████████████████
  [3] GOLD           5816  (14.5%)  ████████████████████████████████████████
  [4] PLATINUM       5857  (14.6%)  ████████████████████████████████████████
  [5] EMERALD        5856  (14.6%)  ████████████████████████████████████████
  [6] DIAMOND        5841  (14.6%)  ████████████████████████████████████████


## 12. Validace exportovaného datasetu

In [11]:
# ── Načtení a validace exportovaného CSV ──────────────────────────────────────
print(f"Načítám {EXPORT_CSV_PATH} …")
df_val = pd.read_csv(EXPORT_CSV_PATH)

print(f"\n{'─'*50}")
print(f"TVAR DATASETU: {df_val.shape[0]:,} řádků × {df_val.shape[1]} sloupců")
print(f"{'─'*50}")

# Chybějící hodnoty
null_counts = df_val.isnull().sum()
null_counts = null_counts[null_counts > 0]
if null_counts.empty:
    print("✅ Žádné chybějící hodnoty!")
else:
    print("\n⚠️  Chybějící hodnoty:")
    print(null_counts.to_string())

# Statistiky numerických sloupců
print("\n📊 Deskriptivní statistiky (numerické sloupce):")
numeric_cols = [
    "kills", "deaths", "assists", "kda", "cs_per_min",
    "damage_per_min", "gold_per_min", "deaths_per_min",
    "visionScore", "timePlayed"
]
print(df_val[numeric_cols].describe().round(3).to_string())

# Distribuce cílové proměnné
print(f"\n🎯 Distribuce cílové proměnné (tier_encoded):")
tier_dist = df_val["tier"].value_counts().sort_index()
for tier, cnt in df_val.groupby("tier_encoded")["tier"].first().items():
    count = (df_val["tier_encoded"] == tier).sum()
    pct   = count / len(df_val) * 100
    print(f"  [{tier}] {cnt:<12} {count:>7,}  ({pct:.1f}%)")

# Datové typy
print(f"\n🔍 Datové typy sloupců:")
print(df_val.dtypes.to_string())

print(f"\n✅ Validace dokončena. Dataset připraven pro ML!")

Načítám lol_rank_dataset.csv …

──────────────────────────────────────────────────
TVAR DATASETU: 40,036 řádků × 25 sloupců
──────────────────────────────────────────────────

⚠️  Chybějící hodnoty:
summonerName    39345
role               25

📊 Deskriptivní statistiky (numerické sloupce):
           kills     deaths    assists        kda  cs_per_min  damage_per_min  gold_per_min  deaths_per_min  visionScore  timePlayed
count  40036.000  40036.000  40036.000  40036.000   40036.000       40036.000     40036.000       40036.000    40036.000   40036.000
mean       5.944      6.242      8.124      3.238       5.421         712.942       402.800           0.205       32.510    1763.197
std        4.937      3.505      6.187      3.704       2.687         371.528        93.527           0.108       26.804     503.911
min        0.000      0.000      0.000      0.000       0.000           0.000       147.472           0.000        0.000      66.000
25%        2.000      4.000      4.000      

C:\Users\matya\AppData\Local\Temp\ipykernel_11196\2238711474.py:3: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df_val = pd.read_csv(EXPORT_CSV_PATH)


## 13. Pomocné nástroje – reset checkpointu, stav DB

Spusť jen v případě potřeby diagnostiky nebo resetu.

In [ ]:
def print_db_status(db_path: str = DB_PATH) -> None:
    """Vytiskne aktuální stav všech tabulek v databázi."""
    c = sqlite3.connect(db_path)
    print("=" * 55)
    print("STAV DATABÁZE")
    print("=" * 55)

    # Celkové počty
    for tbl in ["players", "matches", "participants", "processed_matches"]:
        n = c.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
        print(f"  {tbl:<25} {n:>8,} záznamů")

    # Distribuce per tier v participants
    print("\nParticipants per tier (labeled):")
    rows = c.execute("""
        SELECT tier, COUNT(*) FROM participants
        WHERE tier != 'UNKNOWN'
        GROUP BY tier
        ORDER BY CASE tier
            WHEN 'IRON'     THEN 1 WHEN 'BRONZE'  THEN 2
            WHEN 'SILVER'   THEN 3 WHEN 'GOLD'    THEN 4
            WHEN 'PLATINUM' THEN 5 WHEN 'EMERALD' THEN 6
            WHEN 'DIAMOND'  THEN 7 ELSE 8 END
    """).fetchall()
    for tier, cnt in rows:
        print(f"    {tier:<12} {cnt:>7,}")

    c.close()


def reset_all(confirm: bool = False) -> None:
    """
    ⚠️  NEBEZPEČNÉ: Smaže databázi, checkpoint a log soubor.
    Nastav confirm=True pro skutečné smazání.
    """
    if not confirm:
        print("⚠️  Pro reset nastav: reset_all(confirm=True)")
        return
    for f in [DB_PATH, CHECKPOINT_PATH, LOG_PATH, EXPORT_CSV_PATH]:
        if Path(f).exists():
            Path(f).unlink()
            print(f"  Smazán: {f}")
    print("🔄 Reset dokončen. Znovu spusť buňky 1–3.")


# ── Diagnostika ────────────────────────────────────────────────────────────────
print_db_status()

print("\nCheckpoint stav:")
print(json.dumps(checkpoint.state, indent=2, ensure_ascii=False))

print(f"\nRate limiter – celkem requestů: {rate_limiter.total_requests}")
print(f"Rate limiter – aktuální load:  {rate_limiter.current_load}/{RATE_LIMIT_REQUESTS}")

# reset_all(confirm=True)  # ← odkomentuj POUZE pro úplný reset

NameError: name 'DB_PATH' is not defined